# <font color="#418FDE" size="6.5" uppercase>**Classification Metrics**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Use built-in and custom scorers for classification evaluation. 
- Compute and interpret threshold-based and probability-based classification metrics. 
- Evaluate multiclass, multilabel, ranking, and top-k classification outputs. 


## **1. Scorer API**

### **1.1. Available Scorer Names**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_A/image_01_01.jpg?v=1784037379" width="250">



>* Scorer names select classification performance metrics
>* They standardize cross-validation and model comparison

>* Scorers expect labels, probabilities, or scores
>* Ranking metrics help when thresholds may change

>* Match scorers to class structure and errors
>* Use balanced metrics for imbalanced outcomes



In [ ]:
#@title Python Code - Available Scorer Names

# This script lists classification scorer names.
# Scorer names standardize model evaluation choices.
# The output compares label and probability scorers.

import sklearn
from sklearn.metrics import get_scorer_names

# Scikit-learn provides the official scorer vocabulary.
all_scorers = sorted(get_scorer_names())

# We select common classification scorers for beginners.
chosen_scorers = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision",
]

# These scorers use final predicted class labels.
label_based = ["accuracy", "balanced_accuracy", "precision", "recall", "f1"]

# These scorers use scores or predicted probabilities.
score_based = ["roc_auc", "average_precision"]

# Validate that every teaching scorer exists.
missing_scorers = [name for name in chosen_scorers if name not in all_scorers]
if len(missing_scorers) > 0:
    raise ValueError("A selected scorer name is unavailable.")

print("scikit-learn version:", sklearn.__version__)
print("Available scorer names checked:", len(all_scorers))
print("Label-based examples:", ", ".join(label_based))
print("Score/probability-based examples:", ", ".join(score_based))
print("Use these exact strings in scoring=... for model selection.")



### **1.2. Custom Scorer Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_A/image_01_02.jpg?v=1784037377" width="250">



>* Custom scorers define task-specific success
>* They align evaluation with real-world priorities

>* Match scorers to required prediction outputs
>* Make scores comparable by maximizing better values

>* Validate scorers to avoid misleading model choices
>* Align scoring with real-world costs and risks



In [ ]:
#@title Python Code - Custom Scorer Functions

# This example builds a custom classification scorer.
# The scorer rewards recall more than precision.
# Cross-validation compares accuracy with the custom score.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer
from sklearn.metrics import recall_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged binary classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Build one simple model inside a preprocessing pipeline.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Define a custom scorer from true and predicted labels.
def high_recall_score(y_true, y_pred):
    return recall_score(y_true, y_pred, pos_label=0)

# Convert the metric function into a scikit-learn scorer.
high_recall_scorer = make_scorer(high_recall_score)

# Use the same folds for both scoring rules.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Compare a built-in scorer with the custom scorer.
accuracy_scores = cross_val_score(model, X, y, cv=folds, scoring="accuracy")
custom_scores = cross_val_score(model, X, y, cv=folds, scoring=high_recall_scorer)

# Print concise results for model selection.
print("scikit-learn version:", sklearn.__version__)
print("Mean accuracy:", round(accuracy_scores.mean(), 3))
print("Mean custom high-risk recall:", round(custom_scores.mean(), 3))



### **1.3. Accuracy Variants**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_A/image_01_03.jpg?v=1784037381" width="250">



>* Accuracy counts exact matches across predictions
>* Use it when correctness goals align

>* Balanced accuracy helps with uneven classes
>* It rewards detecting important minority cases

>* Multiclass and multilabel accuracy differ.
>* Choose scorers matching real decision needs.



In [ ]:
#@title Python Code - Accuracy Variants

# Compare accuracy scorers on imbalanced classification data.
# Balanced accuracy values minority classes more fairly.
# The printed scores reveal different model preferences.

import sklearn
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import get_scorer
from sklearn.model_selection import train_test_split

# Create a small imbalanced binary classification dataset.
features, labels = make_classification(
    n_samples=800,
    n_features=6,
    weights=[0.9, 0.1],
    random_state=42,
)

# Check that the dataset has the expected two classes.
class_count = len(set(labels))
if class_count != 2:
    raise ValueError("Expected exactly two classes for this lesson.")

# Split data while preserving the class imbalance.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.3,
    stratify=labels,
    random_state=42,
)

# Train one simple baseline and one simple learned model.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train, y_train)

# Predict labels for both models on the same test set.
baseline_predictions = baseline.predict(X_test)
model_predictions = model.predict(X_test)

# Built-in scorer objects use the same scoring names as model selection.
accuracy_scorer = get_scorer("accuracy")
balanced_scorer = get_scorer("balanced_accuracy")

# Compute direct metric values for beginner-friendly comparison.
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_balanced = balanced_accuracy_score(y_test, baseline_predictions)
model_accuracy = accuracy_score(y_test, model_predictions)
model_balanced = balanced_accuracy_score(y_test, model_predictions)

# Compute the same ideas through the scorer API.
scorer_accuracy = accuracy_scorer(model, X_test, y_test)
scorer_balanced = balanced_scorer(model, X_test, y_test)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Test class counts: class 0={sum(y_test == 0)}, class 1={sum(y_test == 1)}")
print(f"Baseline accuracy: {baseline_accuracy:.3f}")
print(f"Baseline balanced accuracy: {baseline_balanced:.3f}")
print(f"Logistic accuracy: {model_accuracy:.3f}")
print(f"Logistic balanced accuracy: {model_balanced:.3f}")
print(f"Scorer API check: accuracy={scorer_accuracy:.3f}, balanced={scorer_balanced:.3f}")



## **2. Classification Metrics**

### **2.1. Precision Recall F Score**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_A/image_02_01.jpg?v=1784037383" width="250">



>* Precision measures correctness of positive predictions
>* Recall measures how many positives are found

>* Threshold strictness shifts precision and recall
>* Choose thresholds based on error costs

>* F1 balances precision and recall performance
>* Interpret metrics using context and error costs



In [ ]:
#@title Python Code - Precision Recall F Score

# This example compares precision, recall, and F score.
# It uses fixed labels to reveal metric tradeoffs.
# The output shows how stricter predictions change scores.

import numpy as np
from sklearn import __version__ as sklearn_version
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

# True labels mark which cases are actually positive.
y_true = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

# A cautious model predicts positive only three times.
y_pred_cautious = np.array([1, 1, 0, 0, 0, 0, 0, 0, 0, 1])

# An aggressive model predicts positive more often.
y_pred_aggressive = np.array([1, 1, 1, 1, 0, 1, 1, 0, 0, 0])

# Validate that every prediction matches one true label.
if y_true.shape != y_pred_cautious.shape:
    raise ValueError("Cautious predictions must match true labels.")

# Validate the second prediction set too.
if y_true.shape != y_pred_aggressive.shape:
    raise ValueError("Aggressive predictions must match true labels.")

# Compute the three metrics for each prediction style.
model_names = ["cautious", "aggressive"]
prediction_sets = [y_pred_cautious, y_pred_aggressive]

print(f"scikit-learn version: {sklearn_version}")
print("model       precision  recall  f1")

# Each row shows how one prediction style balances errors.
for name, predictions in zip(model_names, prediction_sets):
    precision = precision_score(y_true, predictions, zero_division=0)
    recall = recall_score(y_true, predictions, zero_division=0)
    f1 = f1_score(y_true, predictions, zero_division=0)
    print(f"{name:10s} {precision:9.2f} {recall:7.2f} {f1:4.2f}")

# Count positives to connect the metrics to model behavior.
print("cautious predicts fewer positives, so precision is higher.")
print("aggressive catches more positives, so recall is higher.")



### **2.2. Confusion Matrix Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_A/image_02_02.jpg?v=1784037384" width="250">



>* Compares predicted labels with true labels
>* Shows correct and incorrect classification outcomes

>* Reveals errors hidden by imbalanced accuracy
>* Clarifies false positive and negative tradeoffs

>* Thresholds change confusion matrix outcomes
>* Multiclass matrices reveal confused categories



In [ ]:
#@title Python Code - Confusion Matrix Basics

# This example builds a simple confusion matrix.
# It separates correct predictions from two mistake types.
# The printed counts reveal model behavior clearly.

import numpy as np
import sklearn
from sklearn.metrics import confusion_matrix

# These labels represent a small fraud detection test set.
true_labels = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 0])
predicted_labels = np.array([0, 0, 1, 0, 1, 0, 1, 1, 0, 1])

# The confusion matrix compares true labels with predicted labels.
matrix = confusion_matrix(true_labels, predicted_labels, labels=[0, 1])
true_negative, false_positive, false_negative, true_positive = matrix.ravel()

# These checks keep the example safe and beginner friendly.
if matrix.shape != (2, 2):
    raise ValueError("Expected a two by two confusion matrix.")

# Accuracy alone hides which kind of mistake happened.
accuracy = (true_positive + true_negative) / len(true_labels)
recall = true_positive / (true_positive + false_negative)
precision = true_positive / (true_positive + false_positive)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Confusion matrix rows=true, columns=predicted: {matrix.tolist()}")
print(f"TN={true_negative}, FP={false_positive}, FN={false_negative}, TP={true_positive}")
print(f"Accuracy={accuracy:.2f}, Precision={precision:.2f}, Recall={recall:.2f}")



### **2.3. ROC AUC**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_A/image_02_03.jpg?v=1784037386" width="250">



>* Measures ranking across all classification thresholds
>* Higher AUC means better class separation

>* Compares ranking before choosing a threshold
>* Not accuracy or exact operational performance

>* Imbalance can hide many false alarms
>* Use ROC AUC with other metrics



In [ ]:
#@title Python Code - ROC AUC

# This example demonstrates ROC AUC for classification.
# ROC AUC evaluates ranking across many thresholds.
# The plot shows separation quality visually.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small built-in binary classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the target has exactly two classes.
class_count = len(np.unique(y))
if class_count != 2:
    raise ValueError("This ROC AUC example needs two classes.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Scale features and train one simple classifier.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# ROC AUC uses positive-class scores, not hard labels.
positive_scores = model.predict_proba(X_test)[:, 1]
hard_labels = model.predict(X_test)
auc_from_scores = roc_auc_score(y_test, positive_scores)

# Hard labels lose threshold information and are less informative.
auc_from_labels = roc_auc_score(y_test, hard_labels)
print(f"scikit-learn version: {sklearn.__version__}")
print(f"ROC AUC from probabilities: {auc_from_scores:.3f}")
print(f"ROC AUC from hard labels: {auc_from_labels:.3f}")

# Plot the ROC curve across all decision thresholds.
fig, ax = plt.subplots(figsize=(6, 4))
RocCurveDisplay.from_predictions(y_test, positive_scores, ax=ax)

# Add a random-guessing reference line.
ax.plot([0, 1], [0, 1], linestyle="--", label="Random guessing")
ax.set_title("ROC Curve for Logistic Regression")
ax.legend(loc="lower right")
plt.show()



## **3. Probability Metrics**

### **3.1. Precision Recall Curves**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_A/image_03_01.jpg?v=1784037375" width="250">



>* Evaluate classifiers across changing decision thresholds
>* Balance finding positives against false alarms

>* Highlights positive-class performance in imbalanced data
>* Inspect curves, not just summary scores

>* Evaluate each class or label separately
>* Interpret curves as ranked prediction trade-offs



In [ ]:
#@title Python Code - Precision Recall Curves

# Demonstrate precision recall curves for rare positives.
# Compare thresholds using probabilities from logistic regression.
# Plot the tradeoff and print average precision.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import train_test_split

# Create a small imbalanced binary classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=8,
    n_informative=4,
    weights=[0.9, 0.1],
    random_state=42,
)

# Split data while preserving the rare positive rate.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# Validate that both classes appear in the test set.
unique_test_classes = np.unique(y_test)
if len(unique_test_classes) != 2:
    raise ValueError("The test set must contain both classes.")

# Fit one simple model that can output probabilities.
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Use positive class probabilities as ranking scores.
positive_scores = model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, positive_scores)
average_precision = average_precision_score(y_test, positive_scores)

# Inspect two operating thresholds from the same curve.
strict_threshold = 0.70
permissive_threshold = 0.20
strict_predictions = positive_scores >= strict_threshold
permissive_predictions = positive_scores >= permissive_threshold

# Compute precision and recall safely for each threshold.
strict_precision = np.sum((strict_predictions == 1) & (y_test == 1))
strict_precision = strict_precision / max(np.sum(strict_predictions), 1)
strict_recall = np.sum((strict_predictions == 1) & (y_test == 1))
strict_recall = strict_recall / np.sum(y_test == 1)

permissive_precision = np.sum((permissive_predictions == 1) & (y_test == 1))
permissive_precision = permissive_precision / max(np.sum(permissive_predictions), 1)
permissive_recall = np.sum((permissive_predictions == 1) & (y_test == 1))
permissive_recall = permissive_recall / np.sum(y_test == 1)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Positive class rate in test set: {np.mean(y_test):.2f}")
print(f"Average precision: {average_precision:.3f}")
print(f"Threshold 0.70 -> precision {strict_precision:.2f}, recall {strict_recall:.2f}")
print(f"Threshold 0.20 -> precision {permissive_precision:.2f}, recall {permissive_recall:.2f}")

# Plot the full precision recall tradeoff across thresholds.
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision, label="Logistic regression")
ax.set_title("Precision Recall Curve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend()
plt.show()



### **3.2. Log Loss Brier**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_A/image_03_02.jpg?v=1784037371" width="250">



>* Measures confidence in predicted class probabilities
>* Penalizes confident wrong probability estimates

>* Brier score measures probability prediction error
>* Well-calibrated probabilities match real outcome frequencies

>* Evaluate probabilities beyond predicted labels
>* Compare log loss and Brier reliability



In [ ]:
#@title Python Code - Log Loss Brier

# Compare probability metrics for multiclass predictions.
# Log loss punishes confident wrong probabilities strongly.
# Brier score measures squared probability error.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize

# Load a small built-in multiclass classification dataset.
iris = load_iris()
features = iris.data
target = iris.target

# Check that the example has three classes.
class_labels = np.unique(target)
if len(class_labels) != 3:
    raise ValueError("This lesson expects exactly three classes.")

# Split data while preserving class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, stratify=target, random_state=42
)

# Train one simple model that can predict probabilities.
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)

# Get predicted probabilities for every class.
probabilities = model.predict_proba(X_test)
if probabilities.shape[1] != len(class_labels):
    raise ValueError("Probability columns must match class labels.")

# Create a deliberately overconfident wrong probability table.
wrong_probabilities = probabilities.copy()
wrong_probabilities[:, :] = 0.025
wrong_classes = (y_test + 1) % len(class_labels)
wrong_probabilities[np.arange(len(y_test)), wrong_classes] = 0.95

# Convert true labels into one-hot rows for Brier scoring.
y_test_one_hot = label_binarize(y_test, classes=class_labels)
model_brier = np.mean(np.sum((probabilities - y_test_one_hot) ** 2, axis=1))
wrong_brier = np.mean(np.sum((wrong_probabilities - y_test_one_hot) ** 2, axis=1))

# Compute log loss for both probability sets.
model_log_loss = log_loss(y_test, probabilities, labels=class_labels)
wrong_log_loss = log_loss(y_test, wrong_probabilities, labels=class_labels)

print("scikit-learn version:", sklearn.__version__)
print("Model log loss:", round(model_log_loss, 3))
print("Overconfident wrong log loss:", round(wrong_log_loss, 3))
print("Model multiclass Brier score:", round(model_brier, 3))
print("Overconfident wrong Brier score:", round(wrong_brier, 3))



### **3.3. Multilabel Top K**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_A/image_03_03.jpg?v=1784037373" width="250">



>* Multiple labels can be correct together
>* Top-k checks highest-ranked relevant labels

>* Top-k rewards highly ranked relevant labels
>* Useful when only few suggestions appear

>* Choose k based on application costs
>* Combine top-k metrics with domain review



In [ ]:
#@title Python Code - Multilabel Top K

# Demonstrate multilabel top-k evaluation.
# Compare ranked predictions with true labels.
# Show how k changes recall.

import numpy as np
import sklearn
from sklearn.metrics import top_k_accuracy_score

# Each row can have several correct labels.
label_names = np.array(["dog", "bike", "tree", "person", "car"])
true_labels = np.array([[1, 0, 1, 0, 0], [0, 1, 0, 1, 0], [0, 0, 1, 0, 1]])

# Scores act like model probabilities for each label.
scores = np.array([[0.80, 0.20, 0.70, 0.10, 0.05], [0.30, 0.60, 0.10, 0.55, 0.20], [0.40, 0.10, 0.45, 0.20, 0.50]])

if true_labels.shape != scores.shape:
    raise ValueError("True labels and scores must have matching shapes.")

# Top-k recall counts true labels found in the first k ranks.
print("scikit-learn version:", sklearn.__version__)
print("Example 1 true labels:", ", ".join(label_names[true_labels[0] == 1]))

for k in [1, 2, 3]:
    top_k_indices = np.argsort(scores, axis=1)[:, -k:]
    hits = np.take_along_axis(true_labels, top_k_indices, axis=1).sum(axis=1)
    true_counts = true_labels.sum(axis=1)
    multilabel_top_k_recall = np.mean(hits / true_counts)
    print("Mean multilabel top-" + str(k) + " recall:", round(multilabel_top_k_recall, 3))

# Scikit-learn top-k accuracy is for one true class per row.
single_true_class = np.array([0, 1, 4])
sklearn_top_2 = top_k_accuracy_score(single_true_class, scores, k=2, labels=np.arange(5))
print("Single-label top-2 accuracy:", round(sklearn_top_2, 3))



# <font color="#418FDE" size="6.5" uppercase>**Classification Metrics**</font>


In this lecture, you learned to:
- Use built-in and custom scorers for classification evaluation. 
- Compute and interpret threshold-based and probability-based classification metrics. 
- Evaluate multiclass, multilabel, ranking, and top-k classification outputs. 

In the next Lecture (Lecture B), we will go over 'Regression Clustering'